In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# P2-KN02 setup: repo with the locked picks, the pinned Docling stack of P1-KN06, GPU check
!git clone -q https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /kaggle/working/repo
!cd /kaggle/working/repo && git log --oneline -1
!pip install -q docling==2.143.0 docling-parse==7.24.1 docling-core==2.105.0 docling-ibm-models==4.0.4 pypdf
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# P2-KN02: fetch the 18 locked held-out PDFs (pdfs/heldout_picks.csv) from their sources, the same way P1 did:
# CUAD (HuggingFace), QASPER (arXiv), FinanceBench (its doc_link), OHR (pdfs.zip, as P1-KN03), UniDoc (domain archives,
# as P1-KN01). The three already fetched on the laptop are checked against their catalog sha256. Names, sizes, hashes only.
import hashlib
import os
import sys
import tarfile
import zipfile
from pathlib import Path

import pandas as pd
from huggingface_hub import hf_hub_download

REPO = Path("/kaggle/working/repo")
sys.path.insert(0, str(REPO))
from scripts.p1_cuad_download import REPO_ID as CUAD_REPO, pdf_paths_by_title
from scripts.p1_qasper_add import arxiv_metadata, fetch

OUTPUT = Path("/kaggle/working/heldout_pdfs")
OUTPUT.mkdir(parents=True, exist_ok=True)
FINANCEBENCH_LINKS = {"financebench_americanexpress_2022_10k":
                      "https://s26.q4cdn.com/747928648/files/doc_financials/2022/q4/AMEX-10K-4Q22.pdf"}
picks = pd.read_csv(REPO / "pdfs" / "heldout_picks.csv", dtype={"source_title": str})
known = pd.read_csv(REPO / "pdfs" / "catalog.csv").set_index("doc_id")["sha256"]


def save(doc_id, content):
    digest = hashlib.sha256(content).hexdigest()
    (OUTPUT / f"{doc_id}.pdf").write_bytes(content)
    check = "" if doc_id not in known.index else " | laptop copy: " + ("match" if known[doc_id] == digest else "DIFFERENT")
    print(f"  {doc_id}: {len(content):,} bytes | sha256 {digest[:16]}{check}")


def of(source):
    return picks[picks["source_dataset"] == source].itertuples(index=False)


cuad_paths = pdf_paths_by_title()
for pick in of("CUAD v1"):
    candidates = cuad_paths.get(pick.source_title, [])
    if len(candidates) != 1:
        print(f"  {pick.source_title}: {len(candidates)} matching PDFs -> SKIPPED")
        continue
    save(pick.doc_id, Path(hf_hub_download(CUAD_REPO, candidates[0], repo_type="dataset", local_dir="/tmp/cuad")).read_bytes())

for pick in of("QASPER"):
    version = arxiv_metadata(pick.source_title)["arxiv_version"]
    save(pick.doc_id, fetch(f"https://arxiv.org/pdf/{pick.source_title}{version}"))

for pick in of("FinanceBench"):
    save(pick.doc_id, fetch(FINANCEBENCH_LINKS[pick.doc_id]))

archive = hf_hub_download("opendatalab/OHR-Bench", "pdfs.zip", repo_type="dataset", local_dir="/tmp/ohr")
with zipfile.ZipFile(archive) as bundle:
    pdf_names = [name for name in bundle.namelist()
                 if name.lower().endswith(".pdf") and not name.startswith("__MACOSX") and "/._" not in name]
    for pick in of("OHR-Bench v2"):
        matches = [name for name in pdf_names if name == f"{pick.source_title}.pdf" or name.endswith(f"/{pick.source_title}.pdf")]
        if len(matches) != 1:
            print(f"  {pick.source_title}: {len(matches)} matches {matches} -> SKIPPED")
            continue
        save(pick.doc_id, bundle.read(matches[0]))
os.remove(archive)

for domain, domain_picks in picks[picks["source_dataset"] == "UniDoc-Bench"].groupby("domain", sort=False):
    archive = hf_hub_download("Salesforce/UniDoc-Bench", f"{domain}_pdfs.tar.gz", repo_type="dataset", local_dir="/tmp/unidoc")
    with tarfile.open(archive, "r:gz") as tar:
        pdf_members = [member for member in tar.getmembers() if member.name.lower().endswith(".pdf")]
        for pick in domain_picks.itertuples(index=False):
            matches = [member for member in pdf_members if Path(member.name).stem == pick.source_title]
            if len(matches) != 1:
                print(f"  {domain}/{pick.source_title}: {len(matches)} matches -> SKIPPED")
                continue
            save(pick.doc_id, tar.extractfile(matches[0]).read())
    os.remove(archive)

print("\nPDFs ready:", len(list(OUTPUT.glob("*.pdf"))), "of", len(picks))

In [ ]:
!cd /kaggle/working/repo && python -m scripts.p1_extract_docling --input /kaggle/working/heldout_pdfs --output /kaggle/working/extracted_docling --machine kaggle --run-source P2-KN02